# Musicm8 — Complete Song + Custom Lyrics

Each full run gets a new random seed unless `FIXED_SEED` is set. You can now paste **your own exact lyrics** into `CUSTOM_LYRICS`; Musicm8 preserves your words and only splits long lines into shorter singable phrases. Leave it empty for AI-written lyrics.

Recommended section labels: `[Verse]`, `[Build]`, `[Chorus]`, `[Drop]`, `[Breakdown]`, `[Final]`. The lyrics used are always printed immediately before the audio players.

In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK COMPLETE SONG
# ============================================================

import os, sys, json, shutil, subprocess, secrets
from pathlib import Path

IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32

# NEW variation each run. Put a saved number here to reproduce a song.
FIXED_SEED = None
SEED = int(FIXED_SEED) if FIXED_SEED is not None else secrets.randbelow(2_000_000_000)
print(f"🎲 MUSICM8 SONG SEED: {SEED}")

# ============================================================
# ✍️ YOUR LYRICS
# Leave CUSTOM_LYRICS = "" for AI lyrics.
# Otherwise paste your words between the triple quotes.
# Example format:
# [Verse]
# First line here
# Second line here
#
# [Chorus]
# My hook goes here
# My hook goes here
# ============================================================
CUSTOM_LYRICS = r"""
""".strip()

VOCALS = True
VOCAL_STYLE = "expressive contemporary lead vocal, intimate verses, emotional hook, crisp consonants, very clear words, modern UK electronic production"
VOCAL_LANGUAGE = "en"
VOCAL_STEPS = 32
MATCH_ITERS = 48
MATCH_SECONDS = 3.0
FORCE_SOUND_MATCH = False
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
WORK = ROOT / "work"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
AUDIO.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK / "hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=False)

import torch
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

lyrics_input = WORK / "user_lyrics_input.txt"
if CUSTOM_LYRICS.strip():
    lyrics_input.write_text(CUSTOM_LYRICS.strip() + "\n", encoding="utf-8")
    print("✍️ USER LYRICS MODE — your exact words will be used")
else:
    if lyrics_input.exists(): lyrics_input.unlink()
    print("✍️ AI LYRICS MODE")

cmd = [sys.executable, "-u", "ai_producer_workflow.py", "--root", str(ROOT), "--repo", str(REPO), "--idea", IDEA, "--bars", str(BARS), "--seed", str(SEED), "--ai-model", AI_MODEL, "--match-iters", str(MATCH_ITERS), "--match-seconds", str(MATCH_SECONDS), "--vocal-style", VOCAL_STYLE, "--vocal-language", VOCAL_LANGUAGE, "--vocal-steps", str(VOCAL_STEPS)]
if CUSTOM_LYRICS.strip(): cmd += ["--lyrics-file", str(lyrics_input)]
if not VOCALS: cmd.append("--no-vocals")
if FORCE_SOUND_MATCH: cmd.append("--force-sound-match")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
(PROJECT / "song_seed.txt").write_text(str(SEED), encoding="utf-8")

# ALWAYS show the exact words at the bottom of the run, where they are easy to find.
LYRICS_FILE = PROJECT / "lyrics.txt"
print("\n\n================ 📝 LYRICS USED ================\n")
print(LYRICS_FILE.read_text(encoding="utf-8") if LYRICS_FILE.exists() else "No lyrics file found")
print("=================================================\n")
print(f"🌱 THIS SONG'S SEED: {SEED}")
print("To recreate it, set FIXED_SEED to that number.")

from IPython.display import Audio, display
for label, p in [("INSTRUMENTAL", PROJECT/"master_instrumental.wav"), ("RAW VOCAL", PROJECT/"vocals/neural_lead_raw.wav"), ("SCORE-LOCKED VOCAL", PROJECT/"vocals/neural_lead_synced.wav"), ("VOCAL MIX", PROJECT/"vocals/vocal_mix.wav"), ("FINAL SONG", PROJECT/"master.wav")]:
    if p.exists():
        print("\n🎵", label)
        display(Audio(str(p)))


## 🎤 Vocal-only retry

Use this only when the music is good and you want to retry the singer. It keeps the exact lyrics and automatically reuses the song seed.

In [ ]:
import os, sys, subprocess
from pathlib import Path
from IPython.display import Audio, display

ROOT = Path('/content/drive/MyDrive/Musicm8')
REPO = Path('/content/Musicm8')
PROJECT = ROOT / 'work/ai_projects/latest'
VOCAL_STYLE = "expressive contemporary lead vocal, intimate verses, emotional hook, crisp consonants, very clear words, modern UK electronic production"
VOCAL_LANGUAGE = 'en'
VOCAL_STEPS = 32
seed_file = PROJECT / 'song_seed.txt'
SEED = int(seed_file.read_text().strip()) if seed_file.exists() else 42
print('🎲 Reusing song seed:', SEED)

os.environ['UV_CACHE_DIR'] = '/content/musicm8_uv_cache'
os.environ['UV_PYTHON_INSTALL_DIR'] = '/content/musicm8_uv_python'
os.environ['HF_HOME'] = str(ROOT / 'work/hf_cache')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'], check=True)
subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
os.chdir(REPO)

lyrics = PROJECT/'lyrics.txt'
if lyrics.exists():
    print('\n================ 📝 LYRICS USED ================\n')
    print(lyrics.read_text())
    print('=================================================\n')

print('🎤 Retrying vocals only — existing music will NOT be regenerated')
subprocess.run([sys.executable, '-u', 'retry_vocals.py', '--root', str(ROOT), '--repo', str(REPO), '--style', VOCAL_STYLE, '--language', VOCAL_LANGUAGE, '--steps', str(VOCAL_STEPS), '--seed', str(SEED)], check=True)

for label, path in [('RAW NEURAL VOCAL',PROJECT/'vocals/neural_lead_raw.wav'),('SCORE-LOCKED VOCAL',PROJECT/'vocals/neural_lead_synced.wav'),('PROCESSED VOCAL',PROJECT/'vocals/vocal_mix.wav'),('FINAL SONG',PROJECT/'master.wav')]:
    if path.exists():
        print('\n✅', label)
        display(Audio(str(path)))


## Vocal diagnostics

Only use this if vocal generation fails.

In [ ]:
from pathlib import Path
project = Path('/content/drive/MyDrive/Musicm8/work/ai_projects/latest')
status = project/'vocals/vocal_status.json'
log = project/'vocals/vocal_backend.log'
print(status.read_text() if status.exists() else 'No vocal_status.json')
if log.exists():
    print('\n--- EXACT VOCAL BACKEND LOG TAIL ---')
    print('\n'.join(log.read_text(errors='ignore').splitlines()[-120:]))
